# MLflow Hyperparameter Tuning

`Hyperparameter Tuning`:

- the process to **select the optimal configuration settings** for a machine learning model before the training process begins in order to **maximize its predictive accuracy**.


## Environment

In [ ]:
from src.tracking import tracking_uri
import sys
from pathlib import Path

import mlflow
import torch

# data
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

# mlflow instance
mlflow.set_tracking_uri(tracking_uri())

# print versions and mlflow
print("torch     ", torch.__version__)
print("cuda      ", torch.cuda.is_available())
print("tracking  ", tracking_uri())
print("experiments", [e.name for e in mlflow.search_experiments()])

## Split data

Define same split before sweep.

- sweep results are comparable based on the same split


In [ ]:
from src.data_loader import build_split, verify_split, write_data_yaml

# limit for fast smoke run; e.g. 200
# LIMIT = None  # unlimit
LIMIT = 200  # limit 200

# random seed
SEED = 0

# print split
print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=SEED))
print(verify_split(PROCESSED))

names = (RAW / "classes.txt").read_text().split()
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)
print(data_yaml.read_text())

## Define sweep (Grid search)

`Grid search`:

- Exhaustively tests every single **combination in a predefined matrix of values**.


In [ ]:
import yaml

# experiment name
EXPERIMENT = "yolo-plate-detection-sweep"

# load base param config
base_cfg = yaml.safe_load((ROOT / "configs" / "train.yaml").read_text())
base_cfg["project"] = str(ROOT / base_cfg["project"])

# define grid
GRID = [{"epochs": e, "name": f"tune-ep{e}"} for e in (10, 20, 30)]

print(f"experiment {EXPERIMENT}")
for g in GRID:
    print(" ", g)

## Run sweep

- Each config in grid becomes its own mlflow run
- log with mlflow


In [ ]:
from src.tracking import run_sweep

# run sweep with helper function
results = run_sweep(
    grid=GRID,
    base_cfg=base_cfg,
    data_yaml=data_yaml,
    processed_dir=PROCESSED,
    raw_dir=RAW,
    experiment=EXPERIMENT,
    run_name=lambda cfg: f"cpu-ep{cfg['epochs']}-{cfg['imgsz']}px",
)

for r in results:
    print(r)

## List experiment runs

List emperiment runs for comparison.


In [ ]:
from src.tracking import compare_runs

table = compare_runs(EXPERIMENT)
table

visualize historical metrics


In [ ]:
import matplotlib.pyplot as plt

# get historical run metrics
client = mlflow.tracking.MlflowClient()
experiment = mlflow.get_experiment_by_name(EXPERIMENT)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for _, row in runs.iterrows():
    label = row["tags.mlflow.runName"]
    for ax, metric in zip(axes, ("metrics/mAP50B", "metrics/mAP50-95B")):
        history = client.get_metric_history(row["run_id"], metric)
        if history:
            ax.plot([p.step for p in history], [
                    p.value for p in history], marker="o", ms=3, label=label)

for ax, metric in zip(axes, ("mAP50", "mAP50-95")):
    ax.set_xlabel("epoch")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Cost against benefit -- more epochs always helps a little, so the question is
# whether the gain justifies the minutes.
cols = ["tags.mlflow.runName", "params.epochs", "metrics.metrics/mAP50-95B", "metrics.elapsed_seconds"]
cost = runs[[c for c in cols if c in runs.columns]].copy()
cost.columns = [c.split(".", 1)[-1] for c in cost.columns]
cost = cost.sort_values("epochs", key=lambda s: s.astype(int))
cost["min_per_0.01_mAP"] = (
    cost["elapsed_seconds"] / 60 / (cost["metrics/mAP50-95B"] * 100)
).round(2)
cost

Browse the runs at http://127.0.0.1:5000.